## 3 кейс

**В этом кейсе вы будете рассчитывать:**
* retention
* rolling retention
* lifetime
* churn rate
* mau
* wau
* dau

Файлами для работы являются `registrations.csv` и `entries.csv`. В них хранятся данные о регистрациях пользователей и входа на платформу соответственно.

### **Посчитайте Retention 15 дня (в процентах) для пользователей, зарегистрированных в январе**

Cохраните результат в переменную `retention_15_day`

**Примечание:** результат округлите до 5 знаков после запятой

In [45]:
import csv

In [46]:
def is_leap_year(year: int) -> bool:
    return (year % 4 == 0 and year % 100 != 0) or year % 400 == 0


def get_days_in_month(month: int, year: int) -> int:
    if month in (4, 6, 7, 11):
        return 30
    if month == 2:
        return 29 if is_leap_year(year) else 28
    return 31


def get_absolute_days(date_str: str) -> int:
    year, month, day = map(int, date_str.split("-"))
    
    days = day
    for m in range(1, month):
        days += get_days_in_month(m, year)
    return days

In [47]:
users_signed_up_at_jan = {}

with open("registrations.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, joined_at in reader:
        if joined_at.startswith("2021-01"):
            users_signed_up_at_jan[user_id] = joined_at

users = set()
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        joined_at = users_signed_up_at_jan.get(user_id)
        if joined_at is None:
            continue
        joined_at_days = get_absolute_days(joined_at)
        entry_at_days = get_absolute_days(entry_at)

        difference = entry_at_days - joined_at_days
        if difference == 15:
            users.add(user_id)

retention_15_day = round(len(users) / len(users_signed_up_at_jan) * 100, 5)
retention_15_day

54.65116

In [48]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
# Открываем файл с правильными ответами
with open('metrics.txt', 'r') as f:
    answers = f.read().split('\n')

correct_answer = float(answers[0])

try:
    assert retention_15_day == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Rolling-retention 30 дня (в процентах) для пользователей из той же когорты**

Сохраните результат в переменную `rolling_retention`

**Примечание:** результат округлите до 5 знаков после запятой

In [49]:
users_signed_up_at_jan = {}

with open("registrations.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, joined_at in reader:
        if joined_at.startswith("2021-01"):
            users_signed_up_at_jan[user_id] = joined_at

users = set()
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        joined_at = users_signed_up_at_jan.get(user_id)
        if joined_at is None:
            continue
        joined_at_days = get_absolute_days(joined_at)
        entry_at_days = get_absolute_days(entry_at)

        difference = entry_at_days - joined_at_days
        if difference >= 30:
            users.add(user_id)

rolling_retention = round(len(users) / len(users_signed_up_at_jan) * 100, 5)
rolling_retention

29.06977

In [50]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[1])

try:
    assert rolling_retention == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Lifetime по всем пользователям, посчитанный как интеграл от n-day retention**

Сохраните результат в переменную `lifetime`

**Примечание:** результат округлите до 5 знаков после запятой

In [51]:
users = {}
with open("registrations.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)
    for user_id, joined_at in reader:
        users[user_id] = joined_at

user_active_days = {}
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)
    for user_id, entry_at in reader:
        joined_at = users.get(user_id)
        if joined_at is None:
            continue
        joined_at_days = get_absolute_days(joined_at)
        entry_at_days = get_absolute_days(entry_at)

        difference = entry_at_days - joined_at_days

        if user_id not in user_active_days:
            user_active_days[user_id] = set()
        if difference >= 0:
            user_active_days[user_id].add(difference)

lifetime = sum([len(v) for v in user_active_days.values()]) / len(users)

In [52]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[2])

try:
    assert lifetime == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Churn rate 29 дня (в долях), посчитанный по всем пользователям**

Сохраните результат в переменную `churn_29`.  
Если будете считать CR от Retention, ведите расчет от Rolling Retention, таким образом, мы получим всех, кто не заходил в 29 день и после. Ведя расчет от обычного Retention, наоборот - получим только CR в 29 день.


In [53]:
users = {}
with open("registrations.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, joined_at in reader:
        users[user_id] = get_absolute_days(joined_at)

retained_users = set()
max_entry_day = 0
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        joined_at = users.get(user_id)
        if joined_at is None:
            continue
        entry_at_days = get_absolute_days(entry_at)

        difference = entry_at_days - joined_at
        if difference >= 29:
            retained_users.add(user_id)

        if entry_at_days > max_entry_day:
            max_entry_day = entry_at_days

users_by_date = set()
for user_id, joined_at in users.items():
    if max_entry_day - joined_at >= 29:
        users_by_date.add(user_id)

churn_29 = round(1 - len(retained_users) / len(users_by_date), 5)
churn_29

0.49336

In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[3])

try:
    assert churn_29 == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


0.49336

### **Посчитайте Mau, Wau, Dau за последний месяц/неделю/день записей**

Сохраните результат в переменные `dec_mau`, `dec_wau`, `dec_dau` соответственно

**Примечание:** последний месяц записей - декабрь. Поэтому `mau` рассчитываем для декабря (2021 года), для `wau` берем последнюю неделю - с 25 по 31 декабря, и для `dau` соответственно последний день - 31 декабря.

In [60]:
users = set()
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        if entry_at.startswith("2021-12"):
            users.add(user_id)

dec_mau = len(users)
dec_mau

133

In [61]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[4])

try:
    assert dec_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [62]:
users = set()
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        if entry_at.startswith("2021-12") and int(entry_at.split("-")[-1]) in range(25, 32):
            users.add(user_id)
            
dec_wau = len(users)
dec_wau

84

In [63]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[5])

try:
    assert dec_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [64]:
users = set()
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        if entry_at == "2021-12-31":
            users.add(user_id)
            
dec_dau = len(users)
dec_dau

47

In [65]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[6])

try:
    assert dec_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau усредненные**

Сохраните результат в переменные `avg_mau`, `avg_wau`, `avg_dau` соответственно

**Примечание:** результаты округлите до 5 знаков после запятой

In [86]:
months = {}
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        month = entry_at[:7]
        if month not in months:
            months[month] = set()
        months[month].add(user_id)

avg_mau = round(sum(len(v) for v in months.values()) / len(months), 5)
avg_mau

102.58333

In [87]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[7])

try:
    assert avg_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [95]:
weeks = {}
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        week = get_absolute_days(entry_at) // 7
        if week not in weeks:
            weeks[week] = set()
        weeks[week].add(user_id)

avg_wau = round(sum(len(v) for v in weeks.values()) / len(weeks), 5)
avg_wau

89.81132

In [93]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[8])

try:
    assert avg_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Ответы не совпадают


In [80]:
days = {}
with open("entries.csv", mode="r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=";")
    next(reader)

    for user_id, entry_at in reader:
        if entry_at not in days:
            days[entry_at] = set()
        days[entry_at].add(user_id)

avg_dau = round(sum([len(v) for v in days.values()]) / len(days), 5)
avg_dau

40.5589

In [96]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[9])

try:
    assert avg_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
